# NB02 — property chains and transitivity
A chain composes two relations: parent(a,b) and parent(b,c) imply grandParent(a,c).
Transitivity composes a relation with itself; it does not turn parent into ancestor.
We must declare that inclusion explicitly. Expect three parent, three ancestor, and one grandParent fact.

In [ ]:
from gestaltdb.ontology import Ontology, PropertyChain, SubPropertyOf, Transitive, compile_ontology
from gestaltdb.reasoning import DatalogEngine, Fact, ReasoningLimits

plan = compile_ontology(Ontology('chains', [
    PropertyChain('parent', 'parent', 'grandParent'),
    SubPropertyOf('parent', 'ancestor'),
    Transitive('ancestor'),
]))
asserted = [Fact('a', 'parent', 'b'), Fact('b', 'parent', 'c'), Fact('a', 'parent', 'c')]
result = DatalogEngine().evaluate(plan.program, asserted, limits=ReasoningLimits(), collect_proofs=True)
assert result.match(predicate='grandParent') == (Fact('a', 'grandParent', 'c'),)
assert len(result.match(predicate='ancestor')) == 3
assert len(result.facts) == 7
assert len(plan.origins) == len(plan.program.rules) == 3
assert {o.axiom_id for o in plan.origins} == {a.axiom_id for a in plan.ontology.axioms}
assert not result.explain(Fact('a', 'ancestor', 'c')).truncated
for fact in result.facts:
    print(fact)


ancestor(a,c) has both a direct inclusion route and a recursive route through b.
Set semantics stores that logical fact once. A chosen witness is not all supports;
deleting a premise safely requires separate maintenance support accounting.
Exercise: remove the direct parent(a,c) assertion and check that ancestor(a,c) still follows.